# 🟢 MODELO 1: Baseline - Red Neuronal Simple

## 📋 Objetivo del ejercicio
Crear un modelo básico de clasificación de spam usando una red neuronal simple.

## 🎯 Arquitectura
- **Embedding**: Convierte palabras en vectores
- **GlobalAveragePooling**: Promedia todos los embeddings
- **Dense**: Capa de clasificación

## ⚠️ Problema esperado
Este modelo NO captura el orden de las palabras (Bag of Words approach).

---

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import matthews_corrcoef

# Semilla para reproducibilidad
seed = 42
np.random.seed(seed)

In [ ]:
# Leer datos
train = pd.read_csv("/kaggle/input/u-tad-spam-not-spam-2025-edition/train.csv", index_col="row_id")
test = pd.read_csv("/kaggle/input/u-tad-spam-not-spam-2025-edition/test.csv", index_col="row_id")

print(f"📊 Train shape: {train.shape}")
print(f"📊 Test shape: {test.shape}")
train.head()

## 🔧 Preprocesamiento de texto

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf

tf.random.set_seed(seed)

# Separar features y target
X = train['message'].values
y = train['spam_label'].values

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=seed, stratify=y)

print(f"✅ Training samples: {len(X_train)}")
print(f"✅ Validation samples: {len(X_val)}")

In [ ]:
# Parámetros
MAX_WORDS = 5000
MAX_LEN = 100
EMBEDDING_DIM = 32

# Tokenizar
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

# Convertir a secuencias
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_val_seq = tokenizer.texts_to_sequences(X_val)
X_test_seq = tokenizer.texts_to_sequences(test['message'].values)

# Padding
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_LEN, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post')

print(f"✅ Vocabulario: {len(tokenizer.word_index)} palabras")
print(f"✅ Shape final: {X_train_pad.shape}")

## 🏗️ Construcción del Modelo Baseline

### 📚 Teoría:
- **Embedding**: Transforma índices de palabras en vectores densos
- **GlobalAveragePooling1D**: Promedia todos los embeddings → pierde el orden
- **Dense**: Clasificación final

### ⚠️ Limitación:
"not spam" y "spam not" se tratan igual porque se promedian los vectores.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense

model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=EMBEDDING_DIM, input_length=MAX_LEN),
    GlobalAveragePooling1D(),  # ⚠️ Pierde el orden de las palabras
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 🚀 Entrenamiento

In [ ]:
history = model.fit(
    X_train_pad, y_train,
    validation_data=(X_val_pad, y_val),
    epochs=10,
    batch_size=32,
    verbose=1
)

## 📊 Evaluación

In [ ]:
val_loss, val_acc = model.evaluate(X_val_pad, y_val, verbose=0)
print(f"\n📊 Validation Accuracy: {val_acc:.4f}")
print(f"📊 Validation Loss: {val_loss:.4f}")

## 🔮 Predicciones para test

In [ ]:
# Predecir
y_pred_proba = model.predict(X_test_pad)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()

# Crear submission
submission = pd.read_csv("/kaggle/input/u-tad-spam-not-spam-2025-edition/sample_submission.csv")
submission["spam_label"] = y_pred
submission.to_csv('submission_baseline.csv', index=False)

print("✅ Submission creado: submission_baseline.csv")
submission.head()

---

## 📚 Resumen para Examen

### ✅ Ventajas:
- Simple y rápido
- Pocas capas, fácil de entrenar

### ❌ Desventajas:
- **NO captura orden de palabras**
- Bag of Words approach
- No captura contexto secuencial

### 🔄 Siguiente paso:
**Usar LSTM o GRU** para capturar el orden temporal de las palabras.